# Day 6 — Journey Mapping + Topic Modeling

**Phase:** plan's Day 7–8 (journey + topics), running compressed.

### What this notebook does, and what it deliberately does NOT do

Olist has **no pre-purchase behavior** (no sessions, browsing, cart, checkout abandonment).
So there is no honest acquisition funnel. Anyone who builds one from Olist is making it up.

What's real and defensible here:
1. **Fulfillment funnel** from order timestamps: placed → approved → carrier → delivered. Drop-off is real.
2. **Delivery satisfaction** from the full reviewed population (master), not the oversampled scored set.
3. **Retention** = repeat-purchase rate via `customer_unique_id`, also population-level.
4. **Theme overlay** on the 11,997 scored reviews (complaint corpus, negative-heavy by design).
5. **BERTopic** on negative reviews for granular complaint themes.

### Guardrail (carried from Day 5, do not violate)
- Population stats (funnel, retention, satisfaction-by-delivery): use **master (~99K)**.
- "What customers complain about" (themes, topics): use **scored (11,997)**, which is 86% 1–2★ by design.
- Never compute "% of customers unhappy" from the scored set. That number comes from the full review_score distribution only.


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

# Auto-detect project root whether running from notebooks/ or project root
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
assert (ROOT / 'data' / 'processed').exists(), f"data/processed not found from ROOT={ROOT}"

PROC = ROOT / 'data' / 'processed'
RAW  = ROOT / 'data' / 'raw' / 'olist'

master = pd.read_parquet(PROC / 'olist_master.parquet')
scored = pd.read_parquet(PROC / 'olist_reviews_scored.parquet')

print('ROOT:', ROOT)
print('master:', master.shape)
print('scored:', scored.shape)
print()
print('master cols:', list(master.columns))
print()
print('scored cols:', list(scored.columns))

ROOT: C:\Users\akskumari\Desktop\cx-analytics-project
master: (99441, 31)
scored: (11997, 8)

master cols: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_time_days', 'days_vs_estimate', 'is_late', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'response_time_days', 'language', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_count', 'total_payment', 'payment_types', 'main_payment_type', 'item_count', 'total_price', 'total_freight', 'distinct_products']

scored cols: ['order_id', 'review_pt', 'review_en', 'review_score', 'review_clean', 'review_preprocessed', 'sentiment_pred', 'theme_pred']


## 1. Pull funnel timestamps + status straight from raw orders

I don't trust that `master` kept every raw timestamp (it was built for delivery features, not the funnel).
So I reload the four timestamps + `order_status` + `customer_id` from the raw orders CSV. Small file, no guessing.

In [4]:
ts_cols = ['order_purchase_timestamp', 'order_approved_at',
           'order_delivered_carrier_date', 'order_delivered_customer_date']

orders = pd.read_csv(
    RAW / 'olist_orders_dataset.csv',
    usecols=['order_id', 'customer_id', 'order_status'] + ts_cols,
    parse_dates=ts_cols,
)
print(orders.shape)
print(orders['order_status'].value_counts())

(99441, 7)
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## 2. Fulfillment funnel

A row's status is terminal (one value), so I reconstruct *how far each order got* using timestamp presence.
If `order_approved_at` is null, the order never got approved, and so on. This is a genuine drop-off curve,
not a relabelled status count.

In [5]:
stages = {
    '1. Order placed'        : orders['order_purchase_timestamp'].notna(),
    '2. Payment approved'    : orders['order_approved_at'].notna(),
    '3. Handed to carrier'   : orders['order_delivered_carrier_date'].notna(),
    '4. Delivered to customer': orders['order_delivered_customer_date'].notna(),
}

funnel = pd.DataFrame({'stage': list(stages), 'count': [int(v.sum()) for v in stages.values()]})
top = funnel['count'].iloc[0]
funnel['pct_of_top']  = (funnel['count'] / top * 100).round(2)
funnel['pct_of_prev'] = (funnel['count'] / funnel['count'].shift(1) * 100).round(2)
funnel['dropped_from_prev'] = (funnel['count'].shift(1) - funnel['count']).fillna(0).astype(int)
funnel

,stage,count,pct_of_top,pct_of_prev,dropped_from_prev
0,1. Order placed,99441,100.00,NaN,0
1,2. Payment approved,99281,99.84,99.84,160
2,3. Handed to carrier,97658,98.21,98.37,1623
3,4. Delivered to customer,96476,97.02,98.79,1182


### Where do the dropped orders end up?
The gap between "placed" and "delivered" isn't all in flight. Break it down by status so the friction is named, not hand-waved.

In [6]:
not_delivered = orders[orders['order_delivered_customer_date'].isna()]
print(f"Orders never delivered to customer: {len(not_delivered):,} "
      f"({len(not_delivered)/len(orders)*100:.2f}% of all orders)")
print()
print(not_delivered['order_status'].value_counts())

Orders never delivered to customer: 2,965 (2.98% of all orders)

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


## 3. Delivery satisfaction (population, from master)

This is the honest "does delivery hurt happiness" stat. `review_score` lives in master across the full
reviewed population, so it is NOT contaminated by the Day 3 oversampling. `is_late` was engineered Day 2.

In [7]:
# Guard: only orders that actually have a review score and a late flag
dlv = master.dropna(subset=['review_score', 'is_late']).copy()

by_late = (dlv.groupby('is_late')
              .agg(orders=('order_id', 'size'),
                   avg_score=('review_score', 'mean'),
                   pct_1_2_star=('review_score', lambda s: (s <= 2).mean() * 100))
              .round(2))
by_late.index = by_late.index.map({False: 'On time / early', True: 'Late'})
print(by_late)

if 'delivery_time_days' in dlv.columns:
    print()
    print('Avg review_score by delivery-time bucket:')
    buckets = pd.cut(dlv['delivery_time_days'],
                     bins=[-1, 7, 14, 21, 1000],
                     labels=['0-7d', '8-14d', '15-21d', '22d+'])
    print(dlv.groupby(buckets, observed=True)['review_score'].agg(['size', 'mean']).round(2))

                 orders  avg_score  pct_1_2_star
is_late                                         
On time / early   91820       4.21         11.32
Late               6347       2.27         62.38

Avg review_score by delivery-time bucket:
                     size  mean
delivery_time_days             
0-7d                33425  4.41
8-14d               36009  4.29
15-21d              15164  4.10
22d+                10772  3.01


## 4. Retention (population, from raw customers)

`customer_id` is per-order; `customer_unique_id` is the real person. Repeat rate needs the unique id.
Olist is famously near-zero repeat, so don't be shocked, just report it honestly.

In [8]:
cust = pd.read_csv(RAW / 'olist_customers_dataset.csv',
                   usecols=['customer_id', 'customer_unique_id'])

oc = orders[['order_id', 'customer_id']].merge(cust, on='customer_id', how='left')
orders_per_person = oc.groupby('customer_unique_id')['order_id'].nunique()

repeat_rate = (orders_per_person > 1).mean() * 100
print(f"Unique customers: {orders_per_person.size:,}")
print(f"Repeat customers (2+ orders): {(orders_per_person > 1).sum():,} ({repeat_rate:.2f}%)")
print(f"One-time customers: {(orders_per_person == 1).sum():,} ({100-repeat_rate:.2f}%)")
print()
print('Orders-per-customer distribution:')
print(orders_per_person.value_counts().sort_index().head(8))

Unique customers: 96,096
Repeat customers (2+ orders): 2,997 (3.12%)
One-time customers: 93,099 (96.88%)

Orders-per-customer distribution:
order_id
1    93099
2     2745
3      203
4       30
5        8
6        6
7        3
9        1
Name: count, dtype: int64


## 5. Join scored reviews to master  →  theme / sentiment overlay

From here on we're on the **complaint corpus** (11,997 reviews, 86% 1–2★ by design).
Everything below describes *what people complain about*, never *how many people are unhappy*.

In [9]:
keep = ['order_id', 'is_late', 'review_score', 'delivery_time_days']
keep = [c for c in keep if c in master.columns]

j = scored.merge(master[keep], on='order_id', how='inner', suffixes=('', '_m'))
print('joined scored+master:', j.shape)

# review_score may exist in both; reconcile
if 'review_score_m' in j.columns:
    j['review_score'] = j['review_score'].fillna(j['review_score_m'])
    j = j.drop(columns=['review_score_m'])

print()
print('Theme distribution (complaint corpus):')
print(j['theme_pred'].value_counts())
print()
print('Sentiment distribution (complaint corpus):')
print(j['sentiment_pred'].value_counts())

joined scored+master: (11997, 11)

Theme distribution (complaint corpus):
theme_pred
delivery    7510
quality     2957
other        608
service      540
returns      382
Name: count, dtype: int64

Sentiment distribution (complaint corpus):
sentiment_pred
negative    9853
positive    1477
neutral      667
Name: count, dtype: int64


### Theme by delivery outcome
Does the theme mix shift when the order was late? This is the closest thing to a journey overlay the data honestly supports.

In [10]:
if 'is_late' in j.columns:
    theme_by_late = pd.crosstab(j['theme_pred'], j['is_late'], normalize='columns').round(3) * 100
    theme_by_late.columns = theme_by_late.columns.map({False: 'on_time_%', True: 'late_%'})
    print(theme_by_late)

is_late     on_time_%  late_%
theme_pred                   
delivery         57.1    81.7
other             5.0     5.3
quality          30.1     5.8
returns           3.3     2.8
service           4.5     4.4


### Journey-stage proxy (replaces the shelved classifier)

The journey_stage classifier was killed Day 5 (degenerate labels). Instead derive a **coarse stage from theme**.
State plainly in the report that this is a rule-based heuristic, not a learned model.

In [11]:
theme_to_stage = {
    'delivery': 'delivery',
    'quality' : 'post-purchase',
    'returns' : 'post-purchase',
    'service' : 'support',
    'other'   : 'post-purchase',
}
j['stage_proxy'] = j['theme_pred'].map(theme_to_stage).fillna('post-purchase')
print(j['stage_proxy'].value_counts())

stage_proxy
delivery         7510
post-purchase    3947
support           540
Name: count, dtype: int64


## 6. BERTopic on negative reviews

Granular complaint themes inside the broad `delivery` / `quality` buckets.

**SSL note:** BERTopic's default embedder (`all-MiniLM-L6-v2`) downloads from Hugging Face on first run.
`pip-system-certs` (installed Day 1) patches the Windows cert store, so this download should work where
plain pip needed `--trusted-host`. If it still fails, the `except` block falls back to a TF-IDF + KMeans
topic approach so the notebook doesn't dead-end.

In [12]:
neg = j[(j['sentiment_pred'] == 'negative') | (j['review_score'] <= 2)].copy()
docs = neg['review_clean'].dropna().astype(str)
docs = docs[docs.str.split().str.len() >= 3].tolist()   # drop near-empty
print(f"Negative docs for topic modeling: {len(docs):,}")

Negative docs for topic modeling: 10,589


In [13]:
topic_table = None
try:
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer

    vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=5)
    topic_model = BERTopic(
        min_topic_size=25,
        vectorizer_model=vectorizer,
        verbose=True,
    )
    topics, _ = topic_model.fit_transform(docs)
    topic_table = topic_model.get_topic_info()
    print(topic_table.head(15)[['Topic', 'Count', 'Name']])
except Exception as e:
    print('BERTopic path failed, falling back to TF-IDF + KMeans. Reason:')
    print(repr(e))

2026-06-05 10:35:45,999 - BERTopic - Embedding - Transforming documents to embeddings.


BERTopic path failed, falling back to TF-IDF + KMeans. Reason:
OSError("We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like sentence-transformers/all-MiniLM-L6-v2 is not the path to a directory containing a file named config.json.\nCheckout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.")


In [14]:
# Fallback only runs if BERTopic didn't produce a table
if topic_table is None:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.cluster import KMeans

    tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=5, max_features=3000)
    X = tfidf.fit_transform(docs)
    k = 8
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)

    terms = np.array(tfidf.get_feature_names_out())
    rows = []
    for c in range(k):
        centroid = km.cluster_centers_[c]
        top_terms = terms[centroid.argsort()[::-1][:8]]
        rows.append({'cluster': c,
                     'count': int((km.labels_ == c).sum()),
                     'top_terms': ', '.join(top_terms)})
    topic_table = pd.DataFrame(rows).sort_values('count', ascending=False)
    print(topic_table.to_string(index=False))

 cluster  count                                                                                  top_terms
       3   5815                             product, received, delivery, did, delivered, didn, order, time
       5   1612              bought, came, received, product, product came, products, bought product, sent
       4    856  arrived, product arrived, product, hasn arrived, hasn, waiting, broken, arrived defective
       6    552       quality, recommend, don recommend, don, poor, poor quality, quality product, product
       1    542       haven received, haven, received, received product, product, far haven, far, delivery
       2    454          didn receive, receive, didn, receive product, product, did receive, did, evaluate
       0    401 product delivered, delivered, product, date, delivered time, time, date product, purchased
       7    357                          post, post office, office, shipping, pick, product, collect, didn


## 7. Top friction points (numbers attached)

Pull the headline numbers into one place so the report and dashboard quote the same figures.
Edit the f-strings after you've run the cells above and seen the real values.

In [15]:
late_rate   = master['is_late'].mean() * 100
late_avg    = by_late.loc['Late', 'avg_score']
ontime_avg  = by_late.loc['On time / early', 'avg_score']
delivery_share = (j['theme_pred'] == 'delivery').mean() * 100

print('=== FRICTION POINT SUMMARY (fill into report) ===')
print(f"- Late delivery rate (population): {late_rate:.1f}% of orders")
print(f"- Avg review score: {late_avg:.2f} when late vs {ontime_avg:.2f} when on time "
      f"(gap {ontime_avg - late_avg:.2f} stars)")
print(f"- Delivery is {delivery_share:.0f}% of the complaint corpus by theme")
print(f"- Repeat-purchase rate: {repeat_rate:.1f}% (retention is the structural problem, not just delivery)")
print(f"- Fulfillment drop: {len(not_delivered):,} orders "
      f"({len(not_delivered)/len(orders)*100:.1f}%) never reached the customer")
print()
print('Top granular complaint topics: see topic_table above.')

=== FRICTION POINT SUMMARY (fill into report) ===
- Late delivery rate (population): 6.6% of orders
- Avg review score: 2.27 when late vs 4.21 when on time (gap 1.94 stars)
- Delivery is 63% of the complaint corpus by theme
- Repeat-purchase rate: 3.1% (retention is the structural problem, not just delivery)
- Fulfillment drop: 2,965 orders (3.0%) never reached the customer

Top granular complaint topics: see topic_table above.


## 8. Save outputs

In [17]:
funnel.to_parquet(PROC / 'order_funnel.parquet', index=False)
j.to_parquet(PROC / 'reviews_scored_with_delivery.parquet', index=False)

if topic_table is not None:
    topic_table.to_csv(PROC / 'negative_topics.csv', index=False)

print('Saved to', PROC)
print(' - order_funnel.parquet')
print(' - reviews_scored_with_delivery.parquet')
print(' - negative_topics.csv')

Saved to C:\Users\akskumari\Desktop\cx-analytics-project\data\processed
 - order_funnel.parquet
 - reviews_scored_with_delivery.parquet
 - negative_topics.csv


## Done — what to hand to Day 7

- `order_funnel.parquet` feeds the dashboard funnel view.
- `reviews_scored_with_delivery.parquet` is the text+transactional join for VoC + journey overlay.
- `negative_topics.csv` feeds the VoC deep-dive.

**Day 7:** emotion detection on negatives (HF emotion model), then start RFM/CLV (Day 9–10 phase).
Segment-level theme comparison waits until RFM segments exist. Don't fake segments to do it early.